# Hybrid Quantum CNN v2 on MNIST: re-uploading + controlled-rotation quanvolution

Same quanvolutional LeNet as v1; the **quantum kernel circuit is different**:

```
 input 1×28×28 (pixels in [0,1])
   │  2×2 patches, stride 2 → 196 patches / image
   ▼
 QUANV v2  4 qubits, L layers:
   [ RY(πp_k)·RZ(πp_k) encoding → RY(θ) → CRX ring ↻ → RY(θ') → CRX ring ↺ ] × L   → ⟨Z_k⟩, ⟨X_k⟩   ← quantum
   │  8×14×14 quantum feature maps
   ▼
 Conv 5×5 (8→16) → ReLU → MaxPool → FC 400→120→84→10                                               ← classical
```

| | v1 kernel | **v2 kernel (this notebook)** |
|---|---|---|
| Encoding | RY(πp) once | **dense RY(πp)·RZ(πp)**, **re-uploaded before every layer** |
| Variational block | Rot(φ,θ,ω) per qubit | **RY(θ) per qubit** (two sub-layers) |
| Entangler | fixed CNOT ring | **trainable CRX(α)** rings in both directions (Sim et al. 2019, circuit 14 style) |
| Readout | ⟨Z_k⟩ → 4 channels | **⟨Z_k⟩ + ⟨X_k⟩ → 8 channels** (two measurement bases) |
| Params (L=2) | 24 | **32** |

One layer $l$ on patch $\mathbf p$:
$$U_l(\mathbf p)=\prod_{(c,t)\in\mathcal R_\leftarrow}\!\mathrm{CRX}_{c\to t}(\beta_{l})\bigotimes_k R_Y(\theta'_{lk})\prod_{(c,t)\in\mathcal R_\rightarrow}\!\mathrm{CRX}_{c\to t}(\alpha_{l})\bigotimes_k R_Y(\theta_{lk})\bigotimes_k R_Z(\pi p_k)R_Y(\pi p_k)$$
$$|\psi(\mathbf p)\rangle=U_L(\mathbf p)\cdots U_1(\mathbf p)|0\rangle^{\otimes4},\qquad f_k=\langle Z_k\rangle,\ f_{4+k}=\langle X_k\rangle$$

The trainable CRX entanglers let training decide how strongly pixels interact; the X-basis readout picks up phase information that ⟨Z⟩ alone cannot see.
Simulation is exact, batched and differentiable (PyTorch), and validated against PennyLane in section 4a. Comparison models are in section 13.

## 1. Setup

In [ ]:
# !pip install torch torchvision matplotlib scikit-learn tqdm pennylane   # pennylane optional (validation + drawing)
import time, random, math, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.auto import tqdm

%matplotlib inline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## 2. Configuration

In [ ]:
CFG = dict(
    # quanvolution
    q_layers      = 2,       # re-uploading layers in the quanv circuit (16 params each)
    q_trainable   = True,    # False = fixed random circuit (Henderson et al. 2019)
    shots         = None,    # None = exact <Z>; int = finite-shot estimate at eval time
    # training
    epochs        = 10,
    batch_size    = 128,
    lr            = 1e-3,    # classical parameters
    q_lr          = 1e-2,    # quantum circuit parameters
    seed          = 42,
    data_root     = "./data",
    ckpt          = "best_hybrid_qcnn_v2_mnist.pt",
    run_baselines = True,    # train comparison models in section 13
)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(CFG["seed"])
CFG

## 3. MNIST data
Pixels kept in [0,1]: they map directly to rotation angles π·p.

In [ ]:
tf = transforms.ToTensor()
train_ds = datasets.MNIST(CFG["data_root"], train=True,  download=True, transform=tf)
test_ds  = datasets.MNIST(CFG["data_root"], train=False, download=True, transform=tf)

pin = torch.cuda.is_available()
train_dl = DataLoader(train_ds, CFG["batch_size"], shuffle=True,  num_workers=0, pin_memory=pin, drop_last=True)
test_dl  = DataLoader(test_ds,  CFG["batch_size"], shuffle=False, num_workers=0, pin_memory=pin)
print(f"train: {len(train_ds)}  test: {len(test_ds)}  batches/epoch: {len(train_dl)}")

## 4. Quantum circuit simulator v2 (batched, differentiable)

State `(N, 2**n)`, N = batch × patches, wire 0 = most significant bit (PennyLane convention).
* **Data gates** RY(πp)·RZ(πp) differ per patch, so they are applied as batched `(N,2,2)` contractions.
* **CRX(α), control c → target t**: the RX matrix is applied to the target axis of the control=1 half of the state.
* **Readout**: $\langle Z_k\rangle=\sum_j|\psi_j|^2 z_k(j)$ and $\langle X_k\rangle=\mathrm{Re}\sum_j\psi_j^*\psi_{j\oplus 2^{n-1-k}}$ (bit flip on wire k).

In [ ]:
def apply_1q(state, U, wire, n):
    """state (N,2**n); U (2,2) shared or (N,2,2) per-sample, on `wire`."""
    N = state.shape[0]
    s = state.view(N, 2 ** wire, 2, 2 ** (n - wire - 1))
    s = torch.einsum("ij,bajc->baic", U, s) if U.dim() == 2 else torch.einsum("bij,bajc->baic", U, s)
    return s.reshape(N, 2 ** n)

def apply_ctrl(state, U, c, t, n):
    """Controlled-U (2x2, shared): control wire c, target wire t."""
    N = state.shape[0]
    s = state.view((N,) + (2,) * n)
    s0, s1 = s.select(1 + c, 0), s.select(1 + c, 1)
    ax = 1 + (t if t < c else t - 1)
    s1 = torch.movedim(torch.movedim(s1, ax, -1) @ U.transpose(-1, -2), -1, ax)
    return torch.stack([s0, s1], 1 + c).reshape(N, 2 ** n)

def ry(a):
    c, s = torch.cos(a / 2), torch.sin(a / 2)
    return torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2).to(torch.complex64)

def rz(a):
    e = torch.exp(-0.5j * a.to(torch.complex64)); z = torch.zeros_like(e)
    return torch.stack([torch.stack([e, z], -1), torch.stack([z, e.conj()], -1)], -2)

def rx(a):
    c = torch.cos(a / 2).to(torch.complex64); s = (-1j * torch.sin(a / 2)).to(torch.complex64)
    return torch.stack([torch.stack([c, s], -1), torch.stack([s, c], -1)], -2)


class QuanvCircuitV2(nn.Module):
    """4-qubit kernel: [RY(πp)RZ(πp) re-upload -> RY(θ) -> CRX ring fwd -> RY(θ') -> CRX ring bwd] × L
    -> <Z_k>, <X_k>.   patches (N, n) in [0,1] -> features (N, 2n) in [-1,1]."""

    def __init__(self, n_qubits=4, n_layers=2, trainable=True, shots=None):
        super().__init__()
        n, dim = n_qubits, 2 ** n_qubits
        self.n, self.L, self.shots = n, n_layers, shots
        # weights[l] rows: 0 = RY(θ), 1 = CRX fwd (α), 2 = RY(θ'), 3 = CRX bwd (β); one angle per qubit / per gate
        self.weights = nn.Parameter(torch.rand(n_layers, 4, n) * 2 * math.pi, requires_grad=trainable)
        self.ring_fwd = [(i, (i + 1) % n) for i in reversed(range(n))]     # CRX (control, target)
        self.ring_bwd = [(i, (i - 1) % n) for i in range(n)]
        bits = torch.tensor([[(j >> (n - 1 - k)) & 1 for k in range(n)] for j in range(dim)])
        self.register_buffer("zsign", 1.0 - 2.0 * bits.float())                         # (dim, n)
        flips = torch.stack([torch.arange(dim) ^ (1 << (n - 1 - k)) for k in range(n)])  # (n, dim)
        self.register_buffer("flips", flips)

    @property
    def out_features(self):
        return 2 * self.n

    def forward(self, p):
        n, N = self.n, p.shape[0]
        a = math.pi * p
        enc = rz(a) @ ry(a)                                        # (N, n, 2, 2): RY first, then RZ
        state = torch.zeros(N, 2 ** n, dtype=torch.complex64, device=p.device); state[:, 0] = 1
        for l in range(self.L):
            w = self.weights[l]
            for k in range(n): state = apply_1q(state, enc[:, k], k, n)          # data (re-)upload
            Uy = ry(w[0])
            for k in range(n): state = apply_1q(state, Uy[k], k, n)
            Ux = rx(w[1])
            for g, (c, t) in enumerate(self.ring_fwd): state = apply_ctrl(state, Ux[g], c, t, n)
            Uy = ry(w[2])
            for k in range(n): state = apply_1q(state, Uy[k], k, n)
            Ux = rx(w[3])
            for g, (c, t) in enumerate(self.ring_bwd): state = apply_ctrl(state, Ux[g], c, t, n)
        z = (state.real ** 2 + state.imag ** 2) @ self.zsign                       # <Z_k>
        x = torch.stack([(state.conj() * state[:, self.flips[k]]).real.sum(1) for k in range(n)], 1)  # <X_k>
        out = torch.cat([z, x], 1)
        if self.shots and not self.training:                                      # finite-shot emulation
            p1 = ((1 - out) / 2).clamp(0, 1)
            out = 1 - 2 * torch.distributions.Binomial(self.shots, p1).sample() / self.shots
        return out

    def extra_repr(self):
        return (f"n_qubits={self.n}, layers={self.L}, trainable={self.weights.requires_grad}, "
                f"params={self.weights.numel()}, readout=<Z>x{self.n}+<X>x{self.n}, shots={self.shots}")


class QuanvLayer(nn.Module):
    """Quantum convolution v2: 2x2 kernel, stride 2, shared circuit. (B,1,H,W) -> (B,8,H/2,W/2)."""

    def __init__(self, n_layers=2, trainable=True, shots=None, kernel=2, stride=2):
        super().__init__()
        self.k, self.s = kernel, stride
        self.circuit = QuanvCircuitV2(kernel * kernel, n_layers, trainable, shots)
        self.out_channels = self.circuit.out_features

    def forward(self, x):
        B, _, H, W = x.shape
        Ho, Wo = (H - self.k) // self.s + 1, (W - self.k) // self.s + 1
        patches = F.unfold(x, self.k, stride=self.s).transpose(1, 2).reshape(-1, self.k * self.k)
        f = self.circuit(patches)                                                  # (B*Ho*Wo, 8)
        return f.view(B, Ho * Wo, -1).transpose(1, 2).reshape(B, -1, Ho, Wo)

### 4a. Validation against PennyLane (optional)
Outputs (⟨Z⟩ and ⟨X⟩) and gradients compared on random patches. Skipped if PennyLane is not installed.

In [ ]:
try:
    import pennylane as qml

    def make_qnode(circ, interface="torch"):
        n, L = circ.n, circ.L
        dev = qml.device("default.qubit", wires=n)
        @qml.qnode(dev, interface=interface)
        def qnode(p, w):
            for l in range(L):
                for k in range(n): qml.RY(math.pi * p[k], wires=k); qml.RZ(math.pi * p[k], wires=k)
                for k in range(n): qml.RY(w[l, 0, k], wires=k)
                for g, (c, t) in enumerate(circ.ring_fwd): qml.CRX(w[l, 1, g], wires=[c, t])
                for k in range(n): qml.RY(w[l, 2, k], wires=k)
                for g, (c, t) in enumerate(circ.ring_bwd): qml.CRX(w[l, 3, g], wires=[c, t])
            return [qml.expval(qml.PauliZ(k)) for k in range(n)] + [qml.expval(qml.PauliX(k)) for k in range(n)]
        return qnode

    tc = QuanvCircuitV2(4, 2)
    qn = make_qnode(tc)
    p = torch.rand(6, 4, dtype=torch.float64)
    ours = tc(p.float()).detach()
    ref = torch.stack([torch.stack(qn(pi, tc.weights.detach().double())) for pi in p]).float()
    print("max |ours - pennylane| (outputs) =", (ours - ref).abs().max().item())

    w = tc.weights.detach().double().clone().requires_grad_(True)
    torch.stack(qn(p[0], w)).sum().backward()
    tc.zero_grad(); tc(p[:1].float()).sum().backward()
    print("max |grad diff| (weights)       =", (tc.weights.grad - w.grad.float()).abs().max().item())
except ImportError:
    print("PennyLane not installed - validation skipped.")

## 5. Models

In [ ]:
class LeNetBackend(nn.Module):
    """Classical back-end shared by all front-ends: in_ch x 14 x 14 -> 10 logits."""
    def __init__(self, in_ch=4, num_classes=10):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, 16, 5)          # 14 -> 10
        self.fc1, self.fc2, self.fc3 = nn.Linear(16 * 5 * 5, 120), nn.Linear(120, 84), nn.Linear(84, num_classes)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv(x)), 2)   # 10 -> 5
        x = F.relu(self.fc1(x.flatten(1)))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class HybridQCNN(nn.Module):
    """Quanvolution v2 (quantum) front-end + LeNet (classical) back-end."""
    def __init__(self, q_layers=2, q_trainable=True, shots=None, num_classes=10):
        super().__init__()
        self.quanv = QuanvLayer(q_layers, q_trainable, shots)
        self.backend = LeNetBackend(self.quanv.out_channels, num_classes)   # 8 channels
    def features(self, x):
        return self.quanv(x)
    def forward(self, x):
        return self.backend(self.quanv(x))


class ClassicalControl(nn.Module):
    """Same shape/range as the v2 quanv front-end: Conv 1->8, 2x2, stride 2, tanh."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.front = nn.Conv2d(1, 8, 2, stride=2)
        self.backend = LeNetBackend(8, num_classes)
    def features(self, x):
        return torch.tanh(self.front(x))
    def forward(self, x):
        return self.backend(self.features(x))


class LeNet5(nn.Module):
    """Standard classical LeNet-5 (reference)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.backend = LeNetBackend(6, num_classes)
    def forward(self, x):
        return self.backend(F.max_pool2d(F.relu(self.conv1(x)), 2))


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

model = HybridQCNN(CFG["q_layers"], CFG["q_trainable"], CFG["shots"]).to(device)
print(model)
n_q = count_params(model.quanv)
print(f"\nQuantum params: {n_q} | Classical params: {count_params(model) - n_q:,} | Total trainable: {count_params(model):,}")
print(f"Circuits per image: {14*14} (2x2 patches)  |  per batch of {CFG['batch_size']}: {14*14*CFG['batch_size']:,}")

### 5a. Print the v2 quanvolution circuit
Module summary, weights, and the circuit drawn with PennyLane for one real 2×2 patch. Re-run after training to see the trained weights.

In [ ]:
# !pip install pennylane
import pennylane as qml

if "model" not in globals() or not isinstance(model, HybridQCNN):
    model = HybridQCNN(CFG["q_layers"], CFG["q_trainable"], CFG["shots"]).to(device)
    if os.path.exists(CFG["ckpt"]):
        model.load_state_dict(torch.load(CFG["ckpt"], map_location=device)); print(f"Loaded {CFG['ckpt']}\n")

circ = model.quanv.circuit
n, L = circ.n, circ.L
print(model.quanv)
print(f"\nweights {tuple(circ.weights.shape)} = {circ.weights.numel()} params | kernel 2x2, stride 2 | "
      f"{circ.out_features} output channels | Hilbert dim {2**n}")
print(f"CRX fwd ring (control, target): {circ.ring_fwd}\nCRX bwd ring (control, target): {circ.ring_bwd}\n")
names = ["RY(θ)   per qubit", "CRX(α)  per fwd gate", "RY(θ')  per qubit", "CRX(β)  per bwd gate"]
for l in range(L):
    print(f"Layer {l}")
    for r, nm in enumerate(names):
        print(f"  {nm}: {np.round(circ.weights[l, r].detach().cpu().numpy(), 3)}")

dev = qml.device("default.qubit", wires=n)
@qml.qnode(dev)
def quanv_circuit(p, w):
    for l in range(L):
        for k in range(n): qml.RY(np.pi * p[k], wires=k); qml.RZ(np.pi * p[k], wires=k)   # data (re-)upload
        for k in range(n): qml.RY(w[l, 0, k], wires=k)
        for g, (c, t) in enumerate(circ.ring_fwd): qml.CRX(w[l, 1, g], wires=[c, t])
        for k in range(n): qml.RY(w[l, 2, k], wires=k)
        for g, (c, t) in enumerate(circ.ring_bwd): qml.CRX(w[l, 3, g], wires=[c, t])
    return [qml.expval(qml.PauliZ(k)) for k in range(n)] + [qml.expval(qml.PauliX(k)) for k in range(n)]

# a real patch: a 2x2 patch on the stroke edge of test image 0 (mixed pixel values)
img, lbl = test_ds[0]
pt = F.unfold(img.unsqueeze(0), 2, stride=2)[0].T                        # (196, 4)
idx = ((pt > 0).sum(1) * (pt.std(1) + 1e-3)).argmax().item()
p = pt[idx].numpy()
w = circ.weights.detach().cpu().numpy()
print(f"\nPatch #{idx} of test digit (label {lbl}): pixels {np.round(p, 3)}\n")
print(qml.draw(quanv_circuit, decimals=2, max_length=220)(p, w))

fig, ax = qml.draw_mpl(quanv_circuit, decimals=2, style="pennylane")(p, w)
fig.suptitle(f"Quanvolution v2 kernel: {n} qubits (2×2 patch) × {L} layers, readout ⟨Z⟩+⟨X⟩")
fig.savefig("quanv_circuit_v2.png", dpi=150, bbox_inches="tight"); plt.show()

with torch.no_grad():
    ours = circ(torch.tensor(p, device=device).unsqueeze(0))[0].cpu().numpy()
print("\n<Z>,<X> model     :", np.round(ours, 4))
print("<Z>,<X> PennyLane :", np.round(np.array(quanv_circuit(p, w)), 4))

## 6. Quantum feature maps before training

In [ ]:
def show_feature_maps(m, title, k=4):
    imgs = torch.stack([test_ds[i][0] for i in range(k)]).to(device)
    m.eval()
    with torch.no_grad():
        fm = m.features(imgs).cpu()
    C = fm.shape[1]; h = C // 2
    labels = [f"⟨Z{c}⟩" for c in range(h)] + [f"⟨X{c}⟩" for c in range(h)] if C == 8 else [f"ch {c}" for c in range(C)]
    fig, ax = plt.subplots(k, C + 1, figsize=(1.9 * (C + 1), 2.0 * k))
    for r in range(k):
        ax[r, 0].imshow(imgs[r, 0].cpu(), cmap="gray"); ax[r, 0].set_title(f"input ({test_ds[r][1]})", fontsize=9)
        for c in range(C):
            ax[r, c + 1].imshow(fm[r, c], cmap="RdBu", vmin=-1, vmax=1); ax[r, c + 1].set_title(labels[c], fontsize=9)
    for a in ax.flat: a.axis("off")
    plt.suptitle(title); plt.tight_layout(); plt.show()

show_feature_maps(model, "Quanvolution v2 output channels (untrained circuit)")

## 7. Training and evaluation loops

In [ ]:
def train_epoch(m, loader, opt):
    m.train(); tot, correct, n = 0.0, 0, 0
    for x, y in tqdm(loader, leave=False, desc="train"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        out = m(x); loss = F.cross_entropy(out, y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        tot += loss.item() * y.size(0); correct += (out.argmax(1) == y).sum().item(); n += y.size(0)
    return tot / n, 100.0 * correct / n


@torch.no_grad()
def evaluate(m, loader, return_preds=False):
    m.eval(); tot, correct, n, preds, targets = 0.0, 0, 0, [], []
    for x, y in tqdm(loader, leave=False, desc="eval"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        out = m(x); tot += F.cross_entropy(out, y).item() * y.size(0)
        p = out.argmax(1); correct += (p == y).sum().item(); n += y.size(0)
        if return_preds: preds.append(p.cpu()); targets.append(y.cpu())
    res = dict(loss=tot / n, acc=100.0 * correct / n)
    if return_preds: res["preds"], res["targets"] = torch.cat(preds).numpy(), torch.cat(targets).numpy()
    return res


def make_optimizer(m):
    q = [p for n_, p in m.named_parameters() if "quanv" in n_ and p.requires_grad]
    ids = {id(p) for p in q}
    c = [p for p in m.parameters() if p.requires_grad and id(p) not in ids]
    groups = [{"params": c, "lr": CFG["lr"]}] + ([{"params": q, "lr": CFG["q_lr"]}] if q else [])
    return torch.optim.Adam(groups)


def fit(m, epochs, ckpt, tag):
    opt = make_optimizer(m)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist, best = dict(train_loss=[], train_acc=[], test_loss=[], test_acc=[], time=[]), 0.0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = train_epoch(m, train_dl, opt); te = evaluate(m, test_dl); sched.step()
        for k, v in zip(hist, [tl, ta, te["loss"], te["acc"], time.time() - t0]): hist[k].append(v)
        if te["acc"] > best: best = te["acc"]; torch.save(m.state_dict(), ckpt)
        print(f"[{tag}] Ep {ep:02d} | train {tl:.4f} / {ta:.2f}% | test {te['loss']:.4f} / {te['acc']:.2f}% "
              f"| best {best:.2f}% | {hist['time'][-1]:.1f}s")
    print(f"[{tag}] best test accuracy: {best:.2f}%  -> {ckpt}")
    return hist, best

## 8. Train the hybrid QCNN

In [ ]:
hist, best = fit(model, CFG["epochs"], CFG["ckpt"], "Hybrid QCNN")

## 9. Learning curves

In [ ]:
ep = np.arange(1, len(hist["train_loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, hist["train_loss"], "o-", label="train"); ax[0].plot(ep, hist["test_loss"], "s-", label="test")
ax[0].set(title="Loss", xlabel="epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, hist["train_acc"], "o-", label="train"); ax[1].plot(ep, hist["test_acc"], "s-", label="test")
ax[1].set(title="Accuracy (%)", xlabel="epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 10. Final evaluation of best checkpoint

In [ ]:
model.load_state_dict(torch.load(CFG["ckpt"], map_location=device))
res = evaluate(model, test_dl, return_preds=True)
print(f"Hybrid QCNN v2 test accuracy: {res['acc']:.2f}%   loss: {res['loss']:.4f}\n")
print(classification_report(res["targets"], res["preds"], digits=4))

cm = confusion_matrix(res["targets"], res["preds"])
fig, ax = plt.subplots(figsize=(6.5, 5.5)); im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set(xticks=range(10), yticks=range(10), xlabel="predicted", ylabel="true", title="Confusion matrix (Hybrid QCNN v2)")
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.show()

## 11. Quantum feature maps after training

In [ ]:
show_feature_maps(model, "Quanvolution v2 output channels (trained circuit)")

## 12. Robustness to finite measurement shots

In [ ]:
shot_list = [None, 4096, 1024, 256, 64, 16]
acc_S = []
for s in shot_list:
    model.quanv.circuit.shots = s
    acc_S.append(evaluate(model, test_dl)["acc"]); print(f"shots={str(s):>5}  acc={acc_S[-1]:.2f}%")
model.quanv.circuit.shots = CFG["shots"]

plt.figure(figsize=(6, 3.8))
plt.plot(range(len(shot_list)), acc_S, "o-")
plt.xticks(range(len(shot_list)), ["exact" if s is None else str(s) for s in shot_list])
plt.xlabel("shots per ⟨Z⟩ estimate"); plt.ylabel("test accuracy (%)"); plt.grid(alpha=.3)
plt.title("Accuracy vs measurement shots"); plt.tight_layout(); plt.show()

## 13. Comparison models
* **Random fixed quanv v2**: same circuit, weights frozen at random initialisation.
* **Classical control**: Conv 1→8, 2×2, stride 2, tanh; same output shape (8×14×14) and range as quanv v2, so the gap isolates the quantum kernel.
* **Classical LeNet-5**: standard reference.

Set `CFG['run_baselines']=False` to skip.

In [ ]:
if CFG["run_baselines"]:
    results = {"Hybrid QCNN v2 (trainable)": dict(hist=hist, best=best, params=count_params(model),
                                                front=f"quanv-v2 4q×{CFG['q_layers']}L")}
    specs = [
        ("Hybrid QCNN v2 (random fixed)", lambda: HybridQCNN(CFG["q_layers"], q_trainable=False), f"quanv-v2 4q×{CFG['q_layers']}L, frozen"),
        ("Classical control",          lambda: ClassicalControl(),                              "conv 1→8, 2×2/s2 + tanh"),
        ("Classical LeNet-5",          lambda: LeNet5(),                                        "conv 1→6, 5×5 + pool"),
    ]
    for name, ctor, front in specs:
        set_seed(CFG["seed"])
        m = ctor().to(device)
        h, b = fit(m, CFG["epochs"], f"best_v2_{name.split()[0].lower()}_{len(results)}.pt", name)
        results[name] = dict(hist=h, best=b, params=count_params(m), front=front)

    names = list(results)
    print(f"\n{'Model':<33}{'first layer':<28}{'params':>10}{'best acc %':>12}{'s/epoch':>10}")
    for k in names:
        r = results[k]
        print(f"{k:<33}{r['front']:<28}{r['params']:>10,}{r['best']:>12.2f}{np.mean(r['hist']['time']):>10.1f}")

    ep = np.arange(1, CFG["epochs"] + 1)
    plt.figure(figsize=(7.5, 4.3))
    for k, mk in zip(names, ["s-", "d-", "^-", "o-"]):
        plt.plot(ep, results[k]["hist"]["test_acc"], mk, label=k)
    plt.xlabel("epoch"); plt.ylabel("test accuracy (%)"); plt.grid(alpha=.3); plt.legend()
    plt.title("Test accuracy per epoch"); plt.tight_layout(); plt.show()